# A tiny CNN finds a DNA binding motif

*bioinformatics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/dna_binding_cnn)

Half of a set of random DNA sequences carry one degenerate copy of a transcription-factor binding site. A one-layer 1D convolutional network, $z_{nfp} = W_{fkb}\,x_{n,p+k,b}$ then max-pooling, learns with autograd and Adam to tell them apart; read back as a sequence logo, its most important filter turns into the planted motif.

**Vector form**

$$z_{f}(p) = \sum_{k,b} W_{f}(k, b)\, x(p+k, b) + c_f,\qquad \hat y = \sigma\Big(\sum_f v_f \max_p \operatorname{ReLU} z_f(p) + v_0\Big),\qquad \mathcal L = -\big\langle y\log\hat y + (1-y)\log(1-\hat y)\big\rangle$$

**Index (tensor) form**

$$X_{npkb} = x_{n,p+k,b} \qquad z_{nfp} = W_{fkb}\,X_{npkb} + c_f \qquad m_{nf} = \max_p \operatorname{ReLU}(z_{nfp}) \qquad o_n = v_f\,m_{nf} + v_0 \qquad p_{fkb} = \frac{\kappa_{nf}\,X_{n,p^*_{nf},kb}}{\kappa_{nf}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\pi_{kb}$ | `motif_pwm('GGGACTTTCC', strength)` | (W, 4) | planted position weight matrix: the consensus base with probability "strength", the other three share the rest |
| $x_{nlb}$ | `one_hot(seqs, 4)` | (N, ℓ, 4) | sequence $n$ has base $b \in$ {A, C, G, T} at position $l$; label $y_n = 1$ if a site was planted |
| $X_{npkb}$ | `x.unfold(1, w, 1).transpose(2, 3)` | (N, P, w, 4) | every window of $w$ bases (a view, no copy), $P = \ell - w + 1$ |
| $W_{fkb},\ c_f$ | `params["W"], params["c"]` | (F, w, 4), (F,) | trainable filters: a score per base and position, the same shape as a PWM |
| $z_{nfp}$ | `einsum('npkb,fkb->nfp', X, W) + c` | (N, F, P) | convolution: $k, b$ summed, filter $f$ scored at every position $p$ |
| $m_{nf}$ | `relu(z).amax(dim=2)` | (N, F) | global max pooling: the best match anywhere, so the site may sit at any position |
| $o_n$ | `einsum('nf,f->n', m, v) + v0` | (N,) | logit, $f$ summed; trained with binary cross-entropy, autograd and Adam (mini-batches of 100) |
| $p_{fkb}$ | `filter_pwms(x_pos, W, c)` | (F, w, 4) | back to a motif, as DeepBind does: average the best window $p^*_{nf}$ of each positive sequence whose score passes half the top score ($\kappa_{nf} = 1$), $n$ summed |
| $\mathrm{IC}_k$ | `2 + (p * log2(p)).sum(-1)` | (w,) | information content in bits: the height of the logo stack at position $k$ |
| $r$ | `motif_similarity(p_f, π)` | () | correlation of the learned and planted matrices at the best shift; 1 = identical |

A transcription factor binds DNA where it finds a short, slightly variable word, its motif, usually summarised as a position weight matrix (PWM). Scanning a sequence with a PWM is exactly a 1D convolution of the one-hot sequence with a $w \times 4$ filter, which is why convolutional networks suit genomics: the first layer's filters are motif detectors and max-pooling makes the answer independent of where the site is. Here nothing tells the network what the motif is: it only sees labels, and gradient descent shapes its most useful filter into the planted word (watch the logo sharpen from noise). The dashed line is a reference that does know the motif: scanning with the true PWM. Because the planted sites are degenerate and random DNA sometimes looks like a site, even that cannot reach 100%, and the trained network ends close to it. Only the forward strand is used and the background is uniform; real models also scan the reverse complement and must cope with GC content, repeats and noisy experimental labels. This is the idea behind DeepBind (2015), which learned motifs from protein-binding experiments; models such as Basset, Basenji and Enformer stack many more layers (and attention) to predict chromatin and expression from 100-kb sequences. This toy reproduces only the first-layer idea. Try a weaker motif, a short motif (E-box, CACGTG, appears by chance more often), or fewer filters.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/dna_binding_cnn/main.py
"""Main loop: planted motif → labelled sequences → train the 1D CNN → read the filters as logos → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import accuracy, conv, filter_pwms, information, motif_similarity

SEED = 0
TEST_FRACTION = 0.2
BATCH = 100


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    t0 = time.perf_counter()

    # --- data: random DNA, half of it with one degenerate copy of the motif ---
    consensus = tensors.MOTIFS[params["motif"]]
    motif = tensors.motif_pwm(consensus, params["strength"], dev)  # (W, 4)
    n_seq = 600 if fast else params["n_seq"]
    x, y, _ = tensors.make_dataset(n_seq, params["length"], motif, SEED, dev)  # (N, ℓ, 4), (N,)
    n_test = int(TEST_FRACTION * n_seq)
    x_tr, y_tr, x_te, y_te = x[n_test:], y[n_test:], x[:n_test], y[:n_test]
    print(f"{n_seq} sequences of {params['length']} bp ({n_seq - n_test} train / {n_test} test), half with a "
          f"{params['motif']} site {consensus}, consensus base kept with p = {params['strength']}")  # fmt: skip

    # Reference: scan with the TRUE motif (log-odds vs uniform background), best threshold on the test set.
    true_score = conv(x_te, torch.log(motif / 0.25)[None], torch.zeros(1, device=dev))[:, 0].amax(1)  # (N_test,)
    oracle = max(accuracy(true_score - t, y_te) for t in torch.linspace(0, true_score.max().item(), 200).tolist())
    print(f"reference: scanning with the true PWM classifies {oracle:.1%} of the test set correctly")

    # --- train ---
    weights = tensors.init_weights(params["n_filters"], params["width"], SEED, dev)
    epochs = 2 if fast else params["epochs"]
    hist = solve.train(weights, x_tr, y_tr, x_te, y_te, epochs, params["lr"], BATCH, SEED)
    loss0, loss1 = hist["test_loss"][0], hist["test_loss"][-1]
    print(f"Adam, {epochs} epochs × {len(hist['loss']) // epochs} batches: test loss {loss0:.3f} → {loss1:.3f}, "
          f"accuracy train {hist['train_acc'][-1]:.1%}, test {hist['test_acc'][-1]:.1%}")  # fmt: skip

    # --- open the box: which filter matters, and what has it learned? ---
    with torch.no_grad():
        m = torch.relu(conv(x_te, weights["W"], weights["c"])).amax(2)  # (N_test, F)
        importance = weights["v"] * (m[y_te == 1].mean(0) - m[y_te == 0].mean(0))  # (F,)
        best = int(importance.argmax())
        # Read each snapshot of that filter back as a PWM: average its best-matching windows in the positive test set.
        pos = x_te[y_te == 1]
        pwm_snaps = torch.stack([filter_pwms(pos, W, c)[best] for W, c in hist["snapshots"]]).cpu()  # (S, w, 4)
        r_snaps = [motif_similarity(p, motif.cpu())[0] for p in pwm_snaps]
        shift = motif_similarity(pwm_snaps[-1], motif.cpu())[1]  # where the motif sits inside the filter
        r_all = [motif_similarity(p, motif)[0] for p in filter_pwms(pos, weights["W"], weights["c"])]
    bits = information(pwm_snaps[-1]).sum()
    print(f"most important filter: #{best + 1} (importance {importance[best]:.2f}): correlation with the planted "
          f"motif r = {r_snaps[0]:.2f} at step 1 → {r_snaps[-1]:.3f}, {bits:.1f} bits")  # fmt: skip
    print("final r of every filter: " + ", ".join(f"{r:.2f}" for r in r_all))
    print(f"total time {time.perf_counter() - t0:.1f} s")

    return plot.animate(
        motif.cpu(), consensus, pwm_snaps, r_snaps, hist, oracle, best, shift, out_dir / "dna_binding_cnn.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/dna_binding_cnn/equation.py
r"""Binding-site detection → tensor translation.

A DNA sequence of length ℓ is one-hot: x_{nlb} = 1 if sequence n has base b ∈ {A, C, G, T} at position l.
The network (one convolution layer, as in DeepBind):

    z_{nfp} = W_{fkb} x_{n,p+k,b} + c_f          filter f slid along the sequence   (k, b summed)
    m_{nf}  = max_p ReLU(z_{nfp})                 best match of filter f anywhere     (global max pooling)
    o_n     = v_f m_{nf} + v_0                    logit of "contains a binding site"  (f summed)
    L       = −⟨ y_n log σ(o_n) + (1 − y_n) log(1 − σ(o_n)) ⟩_n          binary cross-entropy

A filter W_{fkb} is a scoring matrix over a w-base window, the same object as a position weight matrix
(log-odds per base and position), so after training it can be read back as a sequence logo.
"""

import torch


def windows(x: torch.Tensor, width: int) -> torch.Tensor:
    """All length-w windows: X_{npkb} = x_{n,p+k,b}.   (N, ℓ, 4) → (N, P, w, 4) with P = ℓ − w + 1 (a view, no copy)."""
    return x.unfold(1, width, 1).transpose(2, 3)


def conv(x: torch.Tensor, W: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """z_{nfp} = W_{fkb} x_{n,p+k,b} + c_f.   x: (N, ℓ, 4), W: (F, w, 4) → (N, F, P);  k, b summed."""
    return torch.einsum("npkb,fkb->nfp", windows(x, W.shape[1]), W) + c[None, :, None]


def forward(x: torch.Tensor, params: dict[str, torch.Tensor]) -> torch.Tensor:
    """Logits o_n (N,): convolution → ReLU → max over positions → linear read-out (f summed)."""
    m = torch.relu(conv(x, params["W"], params["c"])).amax(dim=2)  # (N, F)
    return torch.einsum("nf,f->n", m, params["v"]) + params["v0"]


def loss_fn(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Binary cross-entropy with logits (numerically safe form of −y log σ(o) − (1−y) log(1−σ(o)))."""
    return torch.nn.functional.binary_cross_entropy_with_logits(logits, y)


def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Fraction of sequences with the right call (logit > 0 ⇔ predicted binding site)."""
    return ((logits > 0).float() == y).float().mean().item()


def filter_pwms(x: torch.Tensor, W: torch.Tensor, c: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
    """Read the filters back as position probability matrices p_{fkb} (F, w, 4), the way DeepBind/Basset do it.

    For every sequence n and filter f take the best-scoring window; keep it if its score exceeds threshold × the
    filter's top score; p_{fkb} = average of the kept one-hot windows (n summed), plus a small pseudocount.
    """
    X = windows(x, W.shape[1])  # (N, P, w, 4)
    z = torch.einsum("npkb,fkb->nfp", X, W) + c[None, :, None]  # (N, F, P)
    best, pos = z.max(dim=2)  # (N, F)
    keep = (best > threshold * best.amax(0, keepdim=True)) & (best > 0)  # (N, F)
    win = X[torch.arange(x.shape[0], device=x.device)[:, None], pos]  # (N, F, w, 4) best window per (n, f)
    counts = torch.einsum("nf,nfkb->fkb", keep.float(), win) + 0.25  # pseudocount
    return counts / counts.sum(-1, keepdim=True)


def information(pwm: torch.Tensor) -> torch.Tensor:
    """Information content per position, IC_k = 2 + Σ_b p_kb log₂ p_kb  (bits; 0 = uniform, 2 = one base).  (…, w)."""
    return 2 + (pwm * torch.log2(pwm.clamp_min(1e-9))).sum(-1)


def motif_similarity(pwm: torch.Tensor, motif: torch.Tensor) -> tuple[float, int]:
    """Best Pearson-type correlation between the planted motif (W, 4) and the filter PWM (w, 4), over all shifts.

    Both are centred (minus 1/4) so a uniform column contributes nothing; the filter is padded with uniform
    columns, so a filter that holds only part of the motif scores lower. Returns (r, shift).
    """
    W = motif.shape[0]
    pad = torch.full((W - 1, 4), 0.25, device=pwm.device)
    padded = torch.cat([pad, pwm, pad]) - 0.25  # (w + 2W − 2, 4)
    b = (motif - 0.25).flatten()
    scores = [
        (padded[s : s + W].flatten() @ b / (padded[s : s + W].norm() * b.norm() + 1e-12)).item()
        for s in range(padded.shape[0] - W + 1)
    ]
    best = max(range(len(scores)), key=scores.__getitem__)
    return scores[best], best - (W - 1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/dna_binding_cnn/tensors.py
"""Build the concrete tensors: a planted binding motif, random DNA with and without it, and the network weights."""

import torch

BASES = "ACGT"
MOTIFS = {  # consensus binding sites of three well-known transcription factors
    "nfkb": "GGGACTTTCC",  # NF-κB
    "ap1": "TGACTCA",  # AP-1 (Fos/Jun)
    "ebox": "CACGTG",  # E-box (Myc/Max)
}


def motif_pwm(consensus: str, strength: float, device: torch.device) -> torch.Tensor:
    """Position weight matrix π_{kb}, shape (W, 4): the consensus base with probability `strength`, the others share
    the rest. strength = 1 plants the exact word; smaller values give the degenerate sites real factors bind."""
    pwm = torch.full((len(consensus), 4), (1 - strength) / 3)
    for k, base in enumerate(consensus):
        pwm[k, BASES.index(base)] = strength
    return pwm.to(device)


def make_dataset(n_seq: int, length: int, pwm: torch.Tensor, seed: int, device: torch.device):
    """n_seq random sequences (uniform bases); a random half carry one motif instance, drawn from the PWM, at a
    random position. Returns the one-hot sequences x_{nlb} (N, ℓ, 4), the labels y_n (N,) and the motif starts."""
    gen = torch.Generator().manual_seed(seed)
    W = pwm.shape[0]
    seqs = torch.randint(4, (n_seq, length), generator=gen)
    labels = (torch.rand(n_seq, generator=gen) < 0.5).float()
    starts = torch.randint(length - W + 1, (n_seq,), generator=gen)
    site = torch.multinomial(pwm.cpu(), n_seq, replacement=True, generator=gen).T  # (N, W) bases drawn per position
    for n in torch.nonzero(labels).flatten().tolist():
        seqs[n, starts[n] : starts[n] + W] = site[n]
    x = torch.nn.functional.one_hot(seqs, 4).float()
    return x.to(device), labels.to(device), starts.to(device)


def init_weights(n_filters: int, width: int, seed: int, device: torch.device) -> dict[str, torch.Tensor]:
    """Trainable parameters: filters W_{fkb} (F, w, 4), filter biases c_f (F,), read-out v_f (F,) and v_0 ()."""
    gen = torch.Generator().manual_seed(seed)
    params = {
        "W": 0.1 * torch.randn(n_filters, width, 4, generator=gen),
        "c": torch.zeros(n_filters),
        "v": 0.1 * torch.randn(n_filters, generator=gen),
        "v0": torch.zeros(()),
    }
    return {k: p.to(device).requires_grad_(True) for k, p in params.items()}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/dna_binding_cnn/solve.py
"""Train the convolutional network with mini-batch Adam, recording losses, accuracies and filter snapshots."""

import math

import torch

from problems._common.progress import steps

from .equation import accuracy, forward, loss_fn


def train(params, x_tr, y_tr, x_te, y_te, epochs: int, lr: float, batch: int, seed: int, n_snapshots: int = 40):
    """Adam on the binary cross-entropy, with a little L2 on the filters (weight decay). Returns a history dict:

    loss (n_steps,) per mini-batch;  train_acc, test_acc (n_evals,) at the evaluation steps eval_steps;
    snapshots of the filters (W, c) at snap_steps, for turning them into logos later.
    """
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(params.values(), lr=lr)
    n = x_tr.shape[0]
    per_epoch = (n + batch - 1) // batch
    n_steps = epochs * per_epoch
    # Snapshots on a log scale: the filter changes fastest at the start.
    snap_steps = sorted({int(s) - 1 for s in torch.logspace(0, math.log10(n_steps), n_snapshots).round().tolist()})
    hist = {"loss": torch.zeros(n_steps), "eval_steps": [], "train_acc": [], "test_acc": [], "test_loss": []}
    hist["snap_steps"], hist["snapshots"] = snap_steps, []
    step = 0
    for _ in steps(epochs, "epochs"):
        order = torch.randperm(n, generator=gen).to(x_tr.device)
        for b in range(per_epoch):
            idx = order[b * batch : (b + 1) * batch]
            loss = loss_fn(forward(x_tr[idx], params), y_tr[idx]) + 1e-3 * (params["W"] ** 2).sum()
            opt.zero_grad()
            loss.backward()
            opt.step()
            hist["loss"][step] = loss.item()
            if step in snap_steps:
                with torch.no_grad():
                    hist["snapshots"].append((params["W"].detach().clone(), params["c"].detach().clone()))
                    _evaluate(hist, step, params, x_tr, y_tr, x_te, y_te)
            step += 1
    return hist


def _evaluate(hist, step, params, x_tr, y_tr, x_te, y_te) -> None:
    """Accuracy on (up to 2000 of) the training sequences and on the held-out test set."""
    logits_te = forward(x_te, params)
    hist["eval_steps"].append(step)
    hist["train_acc"].append(accuracy(forward(x_tr[:2000], params), y_tr[:2000]))
    hist["test_acc"].append(accuracy(logits_te, y_te))
    hist["test_loss"].append(loss_fn(logits_te, y_te).item())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/dna_binding_cnn/plot.py
"""Animate training: the most important filter turning into the planted motif (as a logo), loss and accuracy."""

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import numpy as np
import torch
from matplotlib.font_manager import FontProperties
from matplotlib.patches import PathPatch
from matplotlib.textpath import TextPath

from problems._common.plotting import frame_indices, save_gif

from .equation import information

BLUE, ORANGE, GREY = "#2563eb", "#ea580c", "#9ca3af"
BASE_COLOURS = {"A": "#16a34a", "C": "#2563eb", "G": "#f59e0b", "T": "#dc2626"}
FONT = FontProperties(family="DejaVu Sans", weight="bold")


def letter(ax, ch: str, x: float, y: float, height: float) -> PathPatch:
    """Draw base `ch` stretched to fill the box [x − 0.45, x + 0.45] × [y, y + height]."""
    path = TextPath((0, 0), ch, size=1, prop=FONT)
    bb = path.get_extents()
    t = mtransforms.Affine2D().translate(-bb.x0, -bb.y0).scale(0.9 / bb.width, height / bb.height)
    t = t.translate(x - 0.45, y)
    return ax.add_patch(PathPatch(t.transform_path(path), color=BASE_COLOURS[ch], lw=0))


def draw_logo(ax, pwm: np.ndarray, x0: int = 0) -> list[PathPatch]:
    """Sequence logo: at each position the letters are stacked, height = p_kb · IC_k (bits), largest on top."""
    ic = information(torch.from_numpy(pwm)).numpy()
    patches = []
    for k in range(pwm.shape[0]):
        y = 0.0
        for b in np.argsort(pwm[k]):  # smallest first, so the dominant base ends on top
            h = pwm[k, b] * ic[k]
            if h > 0.01:
                patches.append(letter(ax, "ACGT"[b], x0 + k, y, h))
            y += h
    return patches


def setup_logo_axis(ax, width: int, title: str) -> None:
    ax.set_xlim(-0.6, width - 0.4)
    ax.set_ylim(0, 2.05)
    ax.set_ylabel("bits", fontsize=8)
    ax.set_xticks(range(width), [str(k + 1) for k in range(width)], fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(False)
    ax.set_title(title, fontsize=10)


def animate(
    motif: torch.Tensor,
    consensus: str,
    pwm_snaps: torch.Tensor,
    r_snaps: list[float],
    hist: dict,
    oracle: float,
    best: int,
    shift: int,
    path: Path,
) -> Path:
    w = pwm_snaps.shape[1]
    loss = hist["loss"].numpy()
    n_steps = len(loss)
    kernel = max(1, n_steps // 50)
    csum = np.concatenate([[0.0], np.cumsum(loss)])
    lo = np.maximum(np.arange(n_steps) + 1 - kernel, 0)
    smooth = (csum[1:] - csum[lo]) / (np.arange(n_steps) + 1 - lo)  # running mean over the last `kernel` batches
    it_smooth = np.arange(1, n_steps + 1)  # step numbers 1, 2, … (log axis below)
    ev = np.array(hist["eval_steps"]) + 1
    snaps = np.array(hist["snap_steps"]) + 1

    fig = plt.figure(figsize=(9.2, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1])
    ax_m = fig.add_subplot(grid[0, 0])
    ax_f = fig.add_subplot(grid[1, 0])
    ax_l = fig.add_subplot(grid[0, 1])
    ax_a = fig.add_subplot(grid[1, 1])

    # --- planted motif (top) and the learned filter (bottom), both as logos ---
    setup_logo_axis(ax_m, w, f"planted motif {consensus}")
    draw_logo(ax_m, motif.numpy(), shift)  # drawn where the trained filter holds it
    setup_logo_axis(ax_f, w, "")
    ax_f.set_xlabel(f"position in the filter (width {w})", fontsize=8)
    logo_patches: list[PathPatch] = []

    # --- loss ---
    ax_l.plot(it_smooth, smooth, color=GREY, lw=1, alpha=0.5)
    (line_l,) = ax_l.plot([], [], color=BLUE, lw=1.6, label="train (batch avg)")
    (line_lt,) = ax_l.plot([], [], "o-", color=ORANGE, ms=2.5, lw=1.2, label="test")
    ax_l.set_xscale("log")
    ax_l.set_xlim(1, n_steps)
    ax_l.set_ylim(0, max(0.75, smooth.max() * 1.05))
    ax_l.set_title("cross-entropy loss", fontsize=10)
    ax_l.legend(loc="upper right", fontsize=7)
    ax_l.tick_params(labelsize=7)

    # --- accuracy ---
    ax_a.axhline(oracle, color=GREY, ls="--", lw=1)
    ax_a.text(1.2, oracle - 0.015, "true-PWM scan", ha="left", va="top", fontsize=7, color="#6b7280")
    (line_a,) = ax_a.plot([], [], "o-", color=BLUE, ms=2.5, lw=1.4, label="train")
    (line_at,) = ax_a.plot([], [], "o-", color=ORANGE, ms=2.5, lw=1.4, label="test")
    ax_a.set_xscale("log")
    ax_a.set_xlim(1, n_steps)
    ax_a.set_ylim(0.4, 1.0)
    ax_a.set_xlabel("Adam step (mini-batch, log scale)", fontsize=8)
    ax_a.set_title("accuracy", fontsize=10)
    ax_a.legend(loc="lower right", fontsize=7)
    ax_a.tick_params(labelsize=7)

    def update(s: int):
        for p in logo_patches:
            p.remove()
        logo_patches[:] = draw_logo(ax_f, pwm_snaps[s].numpy())
        ax_f.set_title(f"filter #{best + 1} read as a PWM, step {snaps[s]}:  r = {r_snaps[s]:.2f}", fontsize=10)
        upto = it_smooth <= snaps[s]
        line_l.set_data(it_smooth[upto], smooth[upto])
        e = ev <= snaps[s]
        line_lt.set_data(ev[e], np.array(hist["test_loss"])[e])
        line_a.set_data(ev[e], np.array(hist["train_acc"])[e])
        line_at.set_data(ev[e], np.array(hist["test_acc"])[e])

    fig.tight_layout()
    frames = frame_indices(len(snaps), 40)  # snapshots are log-spaced in steps
    frames = frames + [frames[-1]] * 8  # hold the final logo
    return save_gif(fig, update, frames, path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'motif': 'nfkb',
    'strength': 0.9,
    'n_seq': 5000,
    'length': 80,
    'n_filters': 8,
    'width': 12,
    'epochs': 20,
    'lr': 0.003
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.dna_binding_cnn")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.dna_binding_cnn.main")
problem_dir = Path("problems/bioinformatics/dna_binding_cnn")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
